# 02 — Sharpe Stability Ratio scoring

Scores every ETF in the DB that has ≥10y of history (first_date ≤ 2016-01-31) by the Sharpe Stability Ratio (Bajo Traver & Rodríguez Domínguez, 2026).

**Definition.** For window τ, let $Z_t = \widehat{SR}_{t,\tau}$ be the annualized rolling Sharpe of daily excess returns. Then

$$\widehat{SSR}(SR^*) = \frac{\bar{Z} - SR^*}{\hat{\sigma}_{HAC}(Z)}$$

where $\hat{\sigma}_{HAC}$ is the Newey-West Bartlett-kernel HAC standard deviation of the rolling-Sharpe series, with bandwidth from Andrews (1991). Using $SR^*=0$ per §3.4 of the paper (keeps elasticities stable).

**Window & period.**  τ = 252 trading days (1y rolling Sharpe); screened period 2016-01-31 → 2026-01-31 (10y lookback from Jan 2026).

**Scoring.** 0..100 = percentile rank of SSR across the eligible universe (100 = most stable risk-adjusted performance).

> Note on hindsight: screening and simulating on the same 10y window is **in-sample**. A clean walk-forward variant would use 2010–2015 to screen and 2016–2026 to hold. Flagged here, handled later.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd

import macro_framework as mf

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 30)

## 1. Score the universe

In [2]:
START = "2016-01-31"
END   = "2026-01-31"
WINDOW = 252   # 1-year rolling Sharpe
SR_STAR = 0.0  # benchmark (paper §3.4 recommendation)
RF_ANNUAL = 0.0  # treat raw returns as excess for cross-ETF comparison

scored = mf.score_universe(start=START, end=END, window=WINDOW, sr_star=SR_STAR, rf_annual=RF_ANNUAL)
print(f"eligible ETFs scored: {len(scored)}")
scored.head(20)

eligible ETFs scored: 106


,rank,symbol,name,category,ssr,score,mean_rolling_sr,sigma_hac,sr_full,n_obs,n_rolling,L_hac
0,1,SWDA.L,iShares Core MSCI World UCITS ETF,world,0.152500,100.000000,1.107585,7.262869,1.017333,2408,2157,151
1,2,XLK,Technology Select Sector SPDR Fund,sector,0.144515,99.056604,1.189602,8.231694,1.008884,2445,2194,152
2,3,VIG,Vanguard Dividend Appreciation ETF,factor,0.142863,98.113208,1.101938,7.713239,0.871977,2445,2194,152
3,4,QQQ,Invesco QQQ Trust,broad,0.126353,97.169811,1.160258,9.182666,0.973722,2445,2194,152
4,5,SMH,VanEck Semiconductor ETF,sector,0.126216,96.226415,1.126533,8.925471,1.099683,2445,2194,152
5,6,USMV,iShares MSCI USA Min Vol Factor ETF,factor,0.125251,95.283019,1.013506,8.091795,0.703380,2445,2194,152
6,7,IWF,iShares Russell 1000 Growth ETF,factor,0.124856,94.339623,1.196184,9.580504,0.946339,2445,2194,152
7,8,SPLG,SPDR Portfolio S&P 500 ETF,broad,0.124184,93.396226,1.148639,9.249497,0.901651,2445,2194,152
8,9,SPY,SPDR S&P 500 ETF Trust,user,0.123735,92.452830,1.128496,9.120258,0.892754,2445,2194,152
9,10,VOO,Vanguard S&P 500 ETF,broad,0.123517,91.509434,1.138244,9.215297,0.892741,2445,2194,152


## 2. Full ranked table (tail)

In [3]:
scored.tail(15)

,rank,symbol,name,category,ssr,score,mean_rolling_sr,sigma_hac,sr_full,n_obs,n_rolling,L_hac
91,92,VCIT,Vanguard Intermediate-Term Corporate Bond ETF,bond,-0.002024,14.150943,-0.029029,14.339278,0.074611,2445,2194,152
92,93,BNDX,Vanguard Total International Bond ETF,bond,-0.003194,13.207547,-0.034329,10.746306,-0.113606,2445,2194,152
93,94,VCSH,Vanguard Short-Term Corporate Bond ETF,bond,-0.003395,12.264151,-0.052067,15.337806,0.091927,2445,2194,152
94,95,MUB,iShares National Muni Bond ETF,bond,-0.006492,11.320755,-0.086743,13.360609,-0.001646,2445,2194,152
95,96,IEI,iShares 3-7 Year Treasury Bond ETF,bond,-0.012875,10.377358,-0.183320,14.238747,-0.091264,2445,2194,152
96,97,AGG,iShares Core US Aggregate Bond ETF,bond,-0.015396,9.433962,-0.198946,12.922078,-0.072397,2445,2194,152
97,98,IEF,iShares 7-10 Year Treasury Bond ETF,bond,-0.017098,8.490566,-0.208875,12.216135,-0.134346,2445,2194,152
98,99,SHY,iShares 1-3 Year Treasury Bond ETF,bond,-0.017233,7.547170,-0.264382,15.341756,-0.076683,2445,2194,152
99,100,BND,Vanguard Total Bond Market ETF,bond,-0.019060,6.603774,-0.245924,12.902334,-0.099195,2445,2194,152
100,101,GOVT,iShares US Treasury Bond ETF,bond,-0.019153,5.660377,-0.230711,12.045655,-0.137812,2445,2194,152


## 3. Score distribution by category

In [4]:
(
    scored.groupby("category")[["score", "ssr", "mean_rolling_sr", "sigma_hac"]]
    .agg(["median", "count"])
    .round(3)
    .sort_values(("score", "median"), ascending=False)
)

score          ssr       mean_rolling_sr       sigma_hac      
           median count median count          median count    median count
category                                                                  
broad      87.736     8  0.121     8           1.123     8     9.250     8
factor     81.132    13  0.115    13           0.965    13     8.092    13
world      78.774     4  0.112     4           1.019     4     8.813     4
smid       70.755     5  0.099     5           0.773     5     7.979     5
sector     65.094    18  0.092    18           0.661    18     8.048    18
intl       50.943    12  0.071    12           0.589    12     8.726    12
user       45.283     4  0.063     4           0.648     4     9.312     4
em         42.453     4  0.059     4           0.526     4     8.974     4
commodity  35.377     8  0.048     8           0.446     8     8.919     8
thematic   28.302     9  0.041     9           0.462     9    10.781     9
real       23.585     3  0.034     3           0.273     3     8.150     3
bond       11.792    18 -0.005    18          -0.069    18    12.147    18

## 4. Persist for downstream use

In [5]:
out = Path.cwd().parent / "data" / "ssr_scores_2016_2026.parquet"
out.parent.mkdir(exist_ok=True)
scored.to_parquet(out, index=False)
print(f"wrote {out.relative_to(Path.cwd().parent)}  ({len(scored)} rows)")

wrote data/ssr_scores_2016_2026.parquet  (106 rows)
